# 06 — Split estratificado por bloques espaciales

El split de la v1 asigna **rásteres completos** a cada partición. Eso garantiza independencia
espacial, pero deja los tres conjuntos con distribuciones de clase muy distintas
(train 56.1 % / val 45.3 % / test 19.1 % de `com_garimpo`), y con solo 2 rásteres en val y 2 en
test. Consecuencia: validación y test no son comparables, y el modelo elegido por val no es el
que mejor generaliza.

Aquí se construye un split alternativo sobre la **rejilla global** ya validada en
`02b_validacion_espacial.ipynb` (352 × 317 celdas, 111 584 chips, sin huecos ni colisiones):

1. Se agrupan los chips en **bloques cuadrados** de la rejilla.
2. Los bloques enteros se reparten entre train/val/test **estratificando por prevalencia** de
   garimpo, de modo que los tres splits queden con un balance de clases parecido.
3. Se descartan los chips de la **costura** entre bloques de splits distintos, para cortar la
   autocorrelación espacial en los bordes.

Todo lo que se escribe va a subcarpetas `v2_bloques/`. **Ningún archivo de la v1 se modifica.**

## 1. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

RUN_NAME = "v2_bloques"

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "01_raw"
INTERMEDIATE_DIR = DATA_DIR / "02_intermediate"
MODEL_INPUT_DIR = DATA_DIR / "05_model_input" / RUN_NAME
REPORT_DIR = DATA_DIR / "08_reporting" / RUN_NAME
FIGURES_DIR = ROOT / "reports" / "figures"

MANIFEST_RAW = RAW_DIR / "manifesto_chips.csv"
MANIFEST_V1 = INTERMEDIATE_DIR / "manifest_with_split.csv"

# Geometría del chip (Sentinel-2, 10 m/píxel)
TILE_SIZE_PX = 128
S2_PIXEL_M = 10
CHIP_SIDE_M = TILE_SIZE_PX * S2_PIXEL_M

# Parámetros del split
BLOCK_SIZE = 32                                     # lado del bloque, en chips
BUFFER_CHIPS = 2                                    # zona de exclusión entre splits distintos
N_ESTRATOS = 5                                      # estratos de prevalencia
FRACCIONES = {"train": 0.70, "val": 0.15, "test": 0.15}
PRIORIDAD = ("test", "val", "train")                # quién conserva sus chips en la costura
SEED = 42

SPLIT_COLORS = {"train": "#4C72B0", "val": "#DD8452", "test": "#55A868"}

MODEL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Corrida            : {RUN_NAME}")
print(f"Bloque             : {BLOCK_SIZE} chips ({BLOCK_SIZE * CHIP_SIDE_M / 1000:.1f} km de lado)")
print(f"Buffer             : {BUFFER_CHIPS} chips ({BUFFER_CHIPS * CHIP_SIDE_M / 1000:.1f} km)")
print(f"Manifiestos nuevos : {MODEL_INPUT_DIR}")
print(f"Reportes nuevos    : {REPORT_DIR}")

## 2. Carga del manifiesto crudo

Se parte de `manifesto_chips.csv` (que trae las bounding boxes) y se adjunta el split de la v1
como columna `split_v1`, solo para poder comparar los dos repartos al final.

In [ ]:
raw = pd.read_csv(MANIFEST_RAW)
v1 = pd.read_csv(MANIFEST_V1)[["png_path", "split"]].rename(columns={"split": "split_v1"})

df = raw.merge(v1, on="png_path", how="left", validate="one_to_one")

print(f"Chips                 : {len(df):,}")
print(f"Rásteres              : {df['tif_source'].nunique()}")
print(f"Con split v1 asignado : {df['split_v1'].notna().sum():,}")
print(f"Prevalencia global    : {df['label_int'].mean() * 100:.1f} % com_garimpo\n")

df.groupby("split_v1").agg(
    n_chips=("png_path", "size"),
    n_rasteres=("tif_source", "nunique"),
    com_garimpo_pct=("label_int", lambda s: round(s.mean() * 100, 1)),
)

## 3. Rejilla global

Misma proyección que en `02b`: los chips tienen todos el mismo tamaño y el mismo origen, así que
se pueden indexar sobre una rejilla entera compartida por los 12 rásteres.

\[
g_x = \operatorname{round}\!\left(\frac{x_{min} - x_{min}^{\text{global}}}{w_{chip}}\right)
\qquad
g_y = \operatorname{round}\!\left(\frac{y_{max}^{\text{global}} - y_{max}}{h_{chip}}\right)
\]

Las dos aserciones del final replican los tests 5a y 5b de la validación espacial: si fallaran,
el resto del notebook no tendría sentido.

In [ ]:
CHIP_W = float((df["xmax"] - df["xmin"]).mean())
CHIP_H = float((df["ymax"] - df["ymin"]).mean())

X0 = float(df["xmin"].min())
Y0 = float(df["ymax"].max())

gx_float = (df["xmin"] - X0) / CHIP_W
gy_float = (Y0 - df["ymax"]) / CHIP_H

df["gx"] = np.rint(gx_float).astype(int)
df["gy"] = np.rint(gy_float).astype(int)

residuo = max(
    float(np.abs(gx_float - df["gx"]).max()),
    float(np.abs(gy_float - df["gy"]).max()),
)

N_X = int(df["gx"].max()) + 1
N_Y = int(df["gy"].max()) + 1
FORMA = (N_Y, N_X)

colisiones = int((df.groupby(["gy", "gx"]).size() > 1).sum())

print(f"Lado del chip    : {CHIP_W:.12f} x {CHIP_H:.12f} deg")
print(f"Rejilla global   : {N_Y} filas x {N_X} columnas = {N_Y * N_X:,} celdas")
print(f"Residuo de encaje: {residuo:.2e} celdas")
print(f"Celdas con más de un chip: {colisiones:,}")

assert residuo < 1e-6, "los chips no encajan en una rejilla común"
assert colisiones == 0, "hay celdas de la rejilla con más de un chip"

## 4. Bloques espaciales

Cada bloque es un cuadrado de `BLOCK_SIZE × BLOCK_SIZE` celdas de la rejilla. Con 32 chips de
lado son unos 41 km, lo bastante grande para que dos bloques vecinos no compartan el mismo
paisaje inmediato.

In [ ]:
def resumen_bloques(frame: pd.DataFrame, lado: int) -> pd.DataFrame:
    """Agrega los chips en bloques cuadrados de la rejilla global."""
    bloques = (
        frame.assign(block_y=frame["gy"] // lado, block_x=frame["gx"] // lado)
        .groupby(["block_y", "block_x"])
        .agg(n_chips=("png_path", "size"), prevalencia=("label_int", "mean"))
        .reset_index()
    )
    bloques["block_id"] = (
        bloques["block_y"].astype(str) + "_" + bloques["block_x"].astype(str)
    )
    return bloques.set_index("block_id")


bloques = resumen_bloques(df, BLOCK_SIZE)

print(f"Bloques           : {len(bloques):,}")
print(
    f"Chips por bloque  : min {bloques['n_chips'].min():,} | "
    f"mediana {int(bloques['n_chips'].median()):,} | max {bloques['n_chips'].max():,}"
)
print(
    f"Prevalencia       : min {bloques['prevalencia'].min():.2f} | "
    f"media {bloques['prevalencia'].mean():.2f} | max {bloques['prevalencia'].max():.2f}"
)
bloques.head()

## 5. Asignación estratificada de bloques

Los bloques se ordenan en estratos según su prevalencia de garimpo. Dentro de cada estrato se
recorren de mayor a menor tamaño y cada bloque va al split con **mayor déficit relativo** de
chips respecto a su objetivo (70 / 15 / 15).

Estratificar corrige el desbalance de la v1; empezar por los bloques grandes evita que el último
bloque desnivele el reparto.

In [ ]:
def asignar_bloques(
    bloques: pd.DataFrame,
    fracciones: dict[str, float],
    n_estratos: int,
    seed: int,
) -> pd.Series:
    """Reparte bloques enteros entre splits equilibrando tamaño y prevalencia."""
    objetivo = {s: f * bloques["n_chips"].sum() for s, f in fracciones.items()}
    acumulado = {s: 0.0 for s in fracciones}
    asignacion: dict[str, str] = {}

    estratos = pd.qcut(
        bloques["prevalencia"].rank(method="first"), n_estratos, labels=False
    )

    for _, grupo in bloques.groupby(estratos):
        orden = grupo.sample(frac=1.0, random_state=seed).sort_values(
            "n_chips", ascending=False, kind="stable"
        )
        for block_id, fila in orden.iterrows():
            elegido = max(
                fracciones,
                key=lambda s: (objetivo[s] - acumulado[s]) / objetivo[s],
            )
            asignacion[block_id] = elegido
            acumulado[elegido] += fila["n_chips"]

    return pd.Series(asignacion, name="split_bloque")

## 6. Buffer en las costuras

Dos bloques vecinos de splits distintos comparten frontera, y los chips pegados a esa frontera
siguen estando espacialmente correlacionados. Se descartan aplicando una prioridad:
**test conserva todos sus chips**, val cede los que estén a menos de `k` celdas de test, y train
cede los que estén a menos de `k` celdas de val o test.

Así el conjunto de evaluación queda intacto y el coste lo paga el entrenamiento, que es el que
puede permitírselo. El cálculo es conservador: se mide contra la asignación original, sin tener
en cuenta que algunos vecinos ya se habrán descartado.

In [ ]:
def conteo_en_ventana(mask: np.ndarray, k: int) -> np.ndarray:
    """Nº de celdas True en la ventana (2k+1)x(2k+1) centrada en cada celda (imagen integral)."""
    integral = np.zeros((mask.shape[0] + 1, mask.shape[1] + 1), dtype=np.int32)
    integral[1:, 1:] = mask.astype(np.int32).cumsum(axis=0).cumsum(axis=1)

    n_y, n_x = mask.shape
    r0 = np.clip(np.arange(n_y) - k, 0, n_y)
    r1 = np.clip(np.arange(n_y) + k + 1, 0, n_y)
    c0 = np.clip(np.arange(n_x) - k, 0, n_x)
    c1 = np.clip(np.arange(n_x) + k + 1, 0, n_x)

    return (
        integral[np.ix_(r1, c1)]
        - integral[np.ix_(r0, c1)]
        - integral[np.ix_(r1, c0)]
        + integral[np.ix_(r0, c0)]
    )


def marcar_costura(
    gy: np.ndarray,
    gx: np.ndarray,
    splits: np.ndarray,
    forma: tuple[int, int],
    k: int,
    prioridad: tuple[str, ...],
) -> np.ndarray:
    """True para los chips que hay que descartar por estar en la costura entre splits."""
    codigos = {s: i + 1 for i, s in enumerate(prioridad)}
    grid = np.zeros(forma, dtype=np.int8)
    grid[gy, gx] = np.array([codigos[s] for s in splits], dtype=np.int8)

    descartar = np.zeros(forma, dtype=bool)
    for i, split in enumerate(prioridad[1:], start=1):
        vecino_superior = np.zeros(forma, dtype=bool)
        for superior in prioridad[:i]:
            vecino_superior |= conteo_en_ventana(grid == codigos[superior], k) > 0
        descartar |= (grid == codigos[split]) & vecino_superior

    return descartar[gy, gx]


def construir_split(
    frame: pd.DataFrame,
    lado: int,
    k: int,
    fracciones: dict[str, float] = FRACCIONES,
    n_estratos: int = N_ESTRATOS,
    seed: int = SEED,
    prioridad: tuple[str, ...] = PRIORIDAD,
) -> tuple[pd.Series, pd.Series]:
    """Split por bloques + marca de descarte, para un tamaño de bloque y buffer dados."""
    asignacion = asignar_bloques(resumen_bloques(frame, lado), fracciones, n_estratos, seed)

    block_id = (frame["gy"] // lado).astype(str) + "_" + (frame["gx"] // lado).astype(str)
    split_bloque = block_id.map(asignacion).rename("split_bloque")

    descartado = marcar_costura(
        frame["gy"].to_numpy(),
        frame["gx"].to_numpy(),
        split_bloque.to_numpy(),
        FORMA,
        k,
        prioridad,
    )
    return split_bloque, pd.Series(descartado, index=frame.index, name="descartado_costura")

## 7. Sensibilidad: tamaño de bloque y buffer

Antes de fijar los parámetros conviene ver qué cuesta cada combinación. Interesan dos cosas:
`pct_descartado` (cuántos chips se pierden en las costuras) y `rango_com_pct` (cuánto se separan
los tres splits en prevalencia de garimpo; cuanto más bajo, más comparables son val y test).

Como referencia, la v1 tiene un rango de **37 puntos porcentuales** entre train y test.

In [ ]:
filas = []
for lado in (16, 24, 32, 48):
    for k in (0, 1, 2, 4):
        split_bloque, descartado = construir_split(df, lado, k)
        conservado = df.assign(split_bloque=split_bloque)[~descartado]
        perfil_combo = conservado.groupby("split_bloque")["label_int"].agg(["size", "mean"])

        fila = {
            "lado_bloque": lado,
            "lado_km": round(lado * CHIP_SIDE_M / 1000, 1),
            "buffer_chips": k,
            "n_bloques": len(resumen_bloques(df, lado)),
            "pct_descartado": round(descartado.mean() * 100, 2),
        }
        for split in FRACCIONES:
            fila[f"n_{split}"] = int(perfil_combo.loc[split, "size"])
            fila[f"com_pct_{split}"] = round(perfil_combo.loc[split, "mean"] * 100, 1)
        fila["rango_com_pct"] = round(
            (perfil_combo["mean"].max() - perfil_combo["mean"].min()) * 100, 1
        )
        filas.append(fila)

sensibilidad = pd.DataFrame(filas)
sensibilidad.to_csv(REPORT_DIR / f"sensibilidad_bloques_{RUN_NAME}.csv", index=False)
sensibilidad

## 8. Split final

Se aplica la combinación elegida en la celda de configuración. Si la tabla anterior sugiere otros
valores, se cambian `BLOCK_SIZE` y `BUFFER_CHIPS` arriba y se reejecuta desde aquí.

In [ ]:
split_bloque, descartado = construir_split(df, BLOCK_SIZE, BUFFER_CHIPS)

df["split_bloque"] = split_bloque
df["descartado_costura"] = descartado
df["split"] = np.where(df["descartado_costura"], "descartado", df["split_bloque"])

final = df[~df["descartado_costura"]].copy()

resumen_final = final.groupby("split").agg(
    n_chips=("png_path", "size"),
    n_rasteres=("tif_source", "nunique"),
    com_garimpo_pct=("label_int", lambda s: round(s.mean() * 100, 1)),
)
resumen_final["sem_garimpo_pct"] = (100 - resumen_final["com_garimpo_pct"]).round(1)
resumen_final["pct_del_total"] = (resumen_final["n_chips"] / len(final) * 100).round(1)

print(f"Chips totales       : {len(df):,}")
print(
    f"Descartados costura : {int(df['descartado_costura'].sum()):,} "
    f"({df['descartado_costura'].mean() * 100:.2f} %)"
)
print(f"Chips utilizables   : {len(final):,}\n")
resumen_final

## 9. Validación del nuevo split

Mismas comprobaciones que en `02b`, adaptadas al reparto por bloques.

In [ ]:
CHECKS: list[dict[str, str]] = []


def registrar(test: str, estado: str, detalle: str) -> None:
    assert estado in {"OK", "AVISO", "FALLO"}, f"Estado inválido: {estado}"
    CHECKS.append({"test": test, "estado": estado, "detalle": detalle})
    print(f"[{estado}] {test} — {detalle}")


celdas_multi = int((final.groupby(["gy", "gx"])["split"].nunique() > 1).sum())
registrar(
    "1. Una celda, un split",
    "OK" if celdas_multi == 0 else "FALLO",
    f"{celdas_multi} celdas de la rejilla reclamadas por más de un split",
)

CODIGOS = {"train": 1, "val": 2, "test": 3}
grid_final = np.zeros(FORMA, dtype=np.int8)
grid_final[final["gy"].to_numpy(), final["gx"].to_numpy()] = final["split"].map(CODIGOS).to_numpy()

costura_residual = 0
for nombre, codigo in CODIGOS.items():
    ajeno = np.zeros(FORMA, dtype=bool)
    for otro, otro_codigo in CODIGOS.items():
        if otro != nombre:
            ajeno |= conteo_en_ventana(grid_final == otro_codigo, BUFFER_CHIPS) > 0
    costura_residual += int(((grid_final == codigo) & ajeno).sum())

registrar(
    "2. Separación en las costuras",
    "OK" if costura_residual == 0 else "FALLO",
    f"{costura_residual} chips con vecino de otro split a menos de "
    f"{BUFFER_CHIPS} chips ({BUFFER_CHIPS * CHIP_SIDE_M:,} m)",
)

prevalencias = final.groupby("split")["label_int"].mean() * 100
rango_v2 = float(prevalencias.max() - prevalencias.min())
registrar(
    "3. Balance de clases entre splits",
    "OK" if rango_v2 <= 5 else "AVISO",
    f"prevalencia com_garimpo entre {prevalencias.min():.1f} % y "
    f"{prevalencias.max():.1f} % (rango {rango_v2:.1f} pp)",
)

rasteres = final.groupby("split")["tif_source"].nunique()
registrar(
    "4. Cobertura de rásteres",
    "OK" if int(rasteres.min()) >= 6 else "AVISO",
    f"cada split cubre entre {int(rasteres.min())} y {int(rasteres.max())} "
    f"de los {df['tif_source'].nunique()} rásteres",
)

proporciones = final["split"].value_counts(normalize=True)
desvio = float((proporciones - pd.Series(FRACCIONES)).abs().max() * 100)
registrar(
    "5. Tamaño de los splits",
    "OK" if desvio <= 3 else "AVISO",
    f"desviación máxima respecto al objetivo 70/15/15: {desvio:.1f} pp",
)

repetidos = int(final.duplicated(subset=["png_path"]).sum())
registrar(
    "6. Manifiestos disjuntos",
    "OK" if repetidos == 0 else "FALLO",
    f"{repetidos} chips repetidos entre splits",
)

checks_df = pd.DataFrame(CHECKS)
checks_df.to_csv(REPORT_DIR / f"split_validation_{RUN_NAME}.csv", index=False)
checks_df

## 10. Comparación v1 (rásteres) vs. v2 (bloques)

In [ ]:
def perfil(frame: pd.DataFrame, columna: str) -> pd.DataFrame:
    return (
        frame.groupby(columna)
        .agg(
            n_chips=("png_path", "size"),
            n_rasteres=("tif_source", "nunique"),
            com_garimpo_pct=("label_int", "mean"),
        )
        .assign(com_garimpo_pct=lambda d: (d["com_garimpo_pct"] * 100).round(1))
        .rename_axis("split")
    )


perfil_v1 = perfil(df, "split_v1").add_suffix("_v1")
perfil_v2 = perfil(final, "split").add_suffix("_v2")
comparacion = perfil_v1.join(perfil_v2, how="outer")
comparacion.to_csv(REPORT_DIR / "v1_vs_v2_split_comparison.csv")

rango_v1 = perfil_v1["com_garimpo_pct_v1"].max() - perfil_v1["com_garimpo_pct_v1"].min()

print("Rango de prevalencia com_garimpo entre splits (menor = val y test más comparables):")
print(f"  v1 por ráster : {rango_v1:.1f} pp")
print(f"  v2 por bloques: {rango_v2:.1f} pp\n")
comparacion

## 11. Figuras

In [ ]:
grid_mapa = np.zeros(FORMA, dtype=np.int8)
grid_mapa[df["gy"].to_numpy(), df["gx"].to_numpy()] = (
    df["split"].map({"train": 1, "val": 2, "test": 3, "descartado": 4}).to_numpy()
)

cmap = mcolors.ListedColormap(
    ["#FFFFFF", SPLIT_COLORS["train"], SPLIT_COLORS["val"], SPLIT_COLORS["test"], "#BBBBBB"]
)

fig, ax = plt.subplots(figsize=(9, 10))
ax.imshow(grid_mapa, cmap=cmap, vmin=0, vmax=4, interpolation="nearest")

for x in range(0, N_X + 1, BLOCK_SIZE):
    ax.axvline(x - 0.5, color="black", lw=0.4, alpha=0.45)
for y in range(0, N_Y + 1, BLOCK_SIZE):
    ax.axhline(y - 0.5, color="black", lw=0.4, alpha=0.45)

handles = [mpatches.Patch(color=SPLIT_COLORS[s], label=s) for s in ("train", "val", "test")]
handles.append(mpatches.Patch(color="#BBBBBB", label=f"descartado (buffer {BUFFER_CHIPS} chips)"))
ax.legend(handles=handles, loc="upper right", framealpha=0.9)

ax.set_title(
    f"Split por bloques espaciales — bloque {BLOCK_SIZE} chips "
    f"({BLOCK_SIZE * CHIP_SIDE_M / 1000:.1f} km)"
)
ax.set_xlabel("columna de la rejilla global")
ax.set_ylabel("fila de la rejilla global")
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"{RUN_NAME}_split_map.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
paneles = [
    ("v1 — split por ráster", perfil_v1, "com_garimpo_pct_v1"),
    ("v2 — split por bloques", perfil_v2, "com_garimpo_pct_v2"),
]

for ax, (titulo, tabla, columna) in zip(axes, paneles):
    orden = [s for s in ("train", "val", "test") if s in tabla.index]
    valores = tabla.loc[orden, columna]
    ax.bar(orden, valores, color=[SPLIT_COLORS[s] for s in orden])
    ax.axhline(
        df["label_int"].mean() * 100,
        color="black", ls="--", lw=1, label="prevalencia global",
    )
    for i, valor in enumerate(valores):
        ax.text(i, valor + 1, f"{valor:.1f} %", ha="center")
    ax.set_title(titulo)
    ax.set_ylabel("% com_garimpo")

axes[0].legend()
plt.suptitle("Balance de clases por split", y=1.03)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"{RUN_NAME}_balance_clases.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Exportar manifiestos

Los tres manifiestos salen con **el mismo esquema de columnas** que los de la v1, para que el
notebook de entrenamiento pueda consumirlos sin cambiar nada más que la ruta.

In [ ]:
COLUMNAS_MANIFIESTO = [
    "png_path",
    "tif_source",
    "label_str",
    "label_int",
    "split",
    "tile_row",
    "tile_col",
    "centroid_lon",
    "centroid_lat",
]

escritos = {}
for split in ("train", "val", "test"):
    subset = final.loc[final["split"] == split, COLUMNAS_MANIFIESTO].reset_index(drop=True)
    ruta = MODEL_INPUT_DIR / f"manifest_{split}.csv"
    subset.to_csv(ruta, index=False)
    escritos[split] = (ruta, len(subset))

completo = final[COLUMNAS_MANIFIESTO + ["gx", "gy", "overlap_ratio", "split_v1"]].copy()
completo["block_y"] = completo["gy"] // BLOCK_SIZE
completo["block_x"] = completo["gx"] // BLOCK_SIZE
ruta_completo = MODEL_INPUT_DIR / f"manifest_full_{RUN_NAME}.csv"
completo.to_csv(ruta_completo, index=False)

resumen_final.to_csv(REPORT_DIR / f"split_summary_{RUN_NAME}.csv")

df.loc[
    df["descartado_costura"],
    ["png_path", "tif_source", "label_int", "gx", "gy", "split_bloque"],
].to_csv(REPORT_DIR / f"chips_descartados_{RUN_NAME}.csv", index=False)

print("Manifiestos escritos:")
for split, (ruta, n) in escritos.items():
    print(f"  {split:<5} {n:>7,} chips  ->  {ruta}")

print(f"\nManifiesto completo : {ruta_completo}")
print(f"Resumen del split   : {REPORT_DIR / f'split_summary_{RUN_NAME}.csv'}")
print(f"Chips descartados   : {REPORT_DIR / f'chips_descartados_{RUN_NAME}.csv'}")
print(f"Validación          : {REPORT_DIR / f'split_validation_{RUN_NAME}.csv'}")
print("\nNingún archivo de la v1 ha sido modificado.")

## 13. Próximo paso

Entrenar las 4 arquitecturas sobre estos manifiestos con `07_train_bloques.ipynb`, que replica
el protocolo de `04c_training_models_tuned.ipynb` sin cambiar el código de entrenamiento: así la
diferencia de resultados es atribuible al split y no a otra cosa.